<a href="https://colab.research.google.com/github/Coding-7allhh/YT_Auto/blob/main/Part_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# BUFFER SHEET-ONLY VIDEO UPLOADER V2.8 — DIRECT LINK / DISCORD COMPATIBLE
# Reads routing data from the existing Google Sheet. Builds virtual video items
# directly from local CSV files under '/content/metadata_files/' using Discord
# or other web URLs, completely bypassing the need for physical MP4 files on Drive!
# ================================================================

import os, sys, json, time, hashlib, subprocess, importlib.util, traceback
from pathlib import Path
from datetime import datetime, timezone

# ---------------- SETTINGS ----------------
SOURCE_ROOT = "/content/drive/MyDrive/Ready_!"
SHEET_ID = "1AXUiyIYjQVsMO99ET3i98OYkgNQaZHmyxx4fUr4SuXY"
SHEET_TAB_INDEX = 0  # first worksheet, matching the previous workflow
SHEET_RANGE = "A:Z"
ENGINE_ROOT = f"{SOURCE_ROOT}/_BUFFER_SHEET_ONLY_UPLOADER"
STATE_DIR = f"{ENGINE_ROOT}/state"
LOG_DIR = f"{ENGINE_ROOT}/logs"
PUBLISH_NOW = True
MAX_VIDEOS = 0  # 0 = all matching videos
SKIP_ALREADY_PUBLISHED = True
REQUEST_TIMEOUT = 90
MAX_CHANNEL_RETRIES = 1  # retry transient failures once

# Local Metadata Directory
METADATA_DIR = "/content/metadata_files"

# ---------------- DEPENDENCIES ----------------
def install_missing(requirements):
    missing=[]
    for pip_name, module_name in requirements:
        try:
            if importlib.util.find_spec(module_name) is None:
                missing.append(pip_name)
        except Exception:
            missing.append(pip_name)
    if missing:
        print("Installing required packages:", ", ".join(missing), flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

install_missing([
    ("gspread", "gspread"),
    ("google-auth", "google.auth"),
    ("google-api-python-client", "googleapiclient"),
    ("requests", "requests"),
    ("pandas", "pandas"),
    ("openpyxl", "openpyxl")
])
import requests
import pandas as pd

# ---------------- DRIVE / AUTH ----------------
def ensure_drive():
    if os.path.isdir("/content/drive/MyDrive"):
        print("Google Drive already mounted.", flush=True)
        return
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        raise RuntimeError(f"Google Drive mount failed: {e}")

def google_services():
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    scopes = [
        "https://www.googleapis.com/auth/spreadsheets.readonly",
        "https://www.googleapis.com/auth/drive",
    ]
    creds, _ = google.auth.default(scopes=scopes)
    return build("drive", "v3", credentials=creds, cache_discovery=False), creds

# ---------------- HELPERS ----------------
def now():
    return datetime.now(timezone.utc).isoformat()

def log(message):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)

def load_json(path, default):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default

def save_json(path, data):
    tmp=path+".tmp"
    with open(tmp,"w",encoding="utf-8") as f:
        json.dump(data,f,ensure_ascii=False,indent=2)
    os.replace(tmp,path)

def read_sheet():
    import gspread
    from google.auth import default
    creds, _ = default(scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"])
    client = gspread.authorize(creds)
    spreadsheet = client.open_by_key(SHEET_ID)
    worksheet = spreadsheet.get_worksheet(SHEET_TAB_INDEX)
    values = worksheet.get(SHEET_RANGE)
    if not values or len(values) < 2:
        raise RuntimeError("Google Sheet is empty or has no data rows.")
    headers=[str(x).strip().lower() for x in values[0]]
    rows=[]
    for raw in values[1:]:
        raw=list(raw)+[""]*max(0,len(headers)-len(raw))
        rows.append({headers[i]:str(raw[i]).strip() for i in range(len(headers))})
    return rows

def build_routes(rows):
    routes={}
    for row in rows:
        folder=(row.get("folder name") or row.get("folder_name") or row.get("folder") or "").strip()
        key=(row.get("api_key") or row.get("buffer_api_key") or row.get("buffer key") or "").strip()
        if not folder or not key:
            continue
        routes.setdefault(folder.casefold(), {"folder":folder,"keys":[]})
        if key not in routes[folder.casefold()]["keys"]:
            routes[folder.casefold()]["keys"].append(key)
    if not routes:
        raise RuntimeError("No valid routing rows found. Required sheet columns: 'folder name' and 'api_key'.")
    return routes

# ---------------- VIRTUAL VIDEOS SCANNING (CSV DRIVEN) ----------------
def discover_virtual_videos_from_csvs(routes):
    """Reads the local CSV files directly to discover virtual videos with Discord/web links"""
    found = []
    if not os.path.exists(METADATA_DIR):
        log(f"Warning: Metadata directory '{METADATA_DIR}' does not exist!")
        return found

    log(f"Scanning local metadata directory for CSVs: '{METADATA_DIR}'")
    csv_files = [f for f in os.listdir(METADATA_DIR) if f.endswith(".csv")]

    for csv_file in csv_files:
        # Try to infer folder/category name from CSV name, e.g. "generated_metadata_Sad.csv" -> "Sad"
        inferred_category = csv_file.replace("generated_metadata_", "").replace(".csv", "").strip()

        matched_route_key = None
        cat_lower = inferred_category.casefold()

        if cat_lower in routes:
            matched_route_key = cat_lower
        else:
            for r_key in routes:
                if r_key in cat_lower or cat_lower in r_key:
                    matched_route_key = r_key
                    break

        if not matched_route_key:
            # Try general search with full csv name
            for r_key in routes:
                if r_key in csv_file.lower():
                    matched_route_key = r_key
                    break

        # FALLBACK: If there's no match but we only have 1 active category route (like 'Fun'), auto-map it!
        if not matched_route_key and len(routes) == 1:
            matched_route_key = list(routes.keys())[0]
            log(f"  Fallback mapping active: Auto-routing '{csv_file}' to the single available sheet category '{matched_route_key}'")

        if not matched_route_key:
            log(f"  Metadata CSV '{csv_file}' (Inferred Category: '{inferred_category}') skipped: no match in Google Sheet routes.")
            continue

        actual_route_name = routes[matched_route_key]["folder"]
        csv_path = os.path.join(METADATA_DIR, csv_file)
        log(f"  Processing metadata file '{csv_file}' for Category '{actual_route_name}'")

        try:
            df = pd.read_csv(csv_path)
            df.columns = [str(col).strip().lower() for col in df.columns]

            count = 0
            for _, row in df.iterrows():
                row_dict = row.to_dict()
                video_name = str(row_dict.get('video name') or row_dict.get('video_name') or row_dict.get('filename') or "").strip()
                if not video_name:
                    continue

                # Retrieve link
                link = str(
                    row_dict.get('video link') or
                    row_dict.get('video_link') or
                    row_dict.get('url') or
                    row_dict.get('link') or
                    row_dict.get('video path') or
                    row_dict.get('video_path') or
                    ""
                ).strip()

                if not link or not link.lower().startswith("http"):
                    continue

                # Add as virtual video
                found.append({
                    "filename": video_name,
                    "folder_category": actual_route_name,
                    "video_url": link,
                    "title": str(row_dict.get('yt title') or row_dict.get('title') or "").strip(),
                    "description": str(
                        row_dict.get('yt description') or
                        row_dict.get('tiktok caption') or
                        row_dict.get('instagram caption') or
                        row_dict.get('fb caption') or
                        row_dict.get('popular captions') or
                        row_dict.get('description') or
                        row_dict.get('caption') or
                        ""
                    ).strip(),
                    "tags": str(
                        row_dict.get('yt tags') or
                        row_dict.get('hashtags (#)') or
                        row_dict.get('tags') or
                        row_dict.get('tag') or
                        row_dict.get('hashtags') or
                        ""
                    ).strip()
                })
                count += 1
            log(f"    Found {count} virtual video link(s) inside '{csv_file}'")
        except Exception as e:
            log(f"    Error reading metadata from '{csv_file}': {e}")

    return found

# ---------------- BUFFER API ----------------
BUFFER_ENDPOINT="https://api.buffer.com"
ACCOUNT_QUERY="query { account { id organizations { id name } } }"
CHANNEL_QUERY="query GetChannels($organizationId: OrganizationId!) { channels(input:{organizationId:$organizationId}) { id name service displayName descriptor externalLink isDisconnected isLocked } }"

YOUTUBE_MUTATION='''mutation CreateYouTubeVideo($channelId: ChannelId!,$text: String!,$videoUrl: String!,$title: String!,$categoryId: String!,$privacy: YoutubePrivacy!) { createPost(input:{channelId:$channelId,text:$text,schedulingType:automatic,mode:shareNow,assets:[{video:{url:$videoUrl}}],metadata:{youtube:{title:$title,categoryId:$categoryId,privacy:$privacy}}}) { ... on PostActionSuccess { post { id text status dueAt assets { id mimeType source } } } ... on MutationError { message } } }'''
INSTAGRAM_MUTATION='''mutation CreateInstagramReel($channelId: ChannelId!,$text: String!,$videoUrl: String!) { createPost(input:{channelId:$channelId,text:$text,schedulingType:automatic,mode:shareNow,assets:[{video:{url:$videoUrl}}],metadata:{instagram:{type:reel,shouldShareToFeed:true}}}) { ... on PostActionSuccess { post { id text status dueAt assets { id mimeType source } } } ... on MutationError { message } } }'''
GENERIC_MUTATION='''mutation CreateVideoPost($channelId: ChannelId!,$text: String!,$videoUrl: String!) { createPost(input:{channelId:$channelId,text:$text,schedulingType:automatic,mode:shareNow,assets:[{video:{url:$videoUrl}}]}) { ... on PostActionSuccess { post { id text status dueAt assets { id mimeType source } } } ... on MutationError { message } } }'''

def buffer_gql(key, query, variables=None):
    response=requests.post(BUFFER_ENDPOINT,headers={"Authorization":f"Bearer {key}","Content-Type":"application/json"},json={"query":query,"variables":variables or {}},timeout=REQUEST_TIMEOUT)
    if response.status_code >= 400:
        raise RuntimeError(f"Buffer HTTP {response.status_code}: {response.text[:400]}")
    data=response.json()
    if data.get("errors"):
        raise RuntimeError(str(data["errors"][:2]))
    if not data.get("data"):
        raise RuntimeError(f"Unexpected Buffer response: {response.text[:400]}")
    return data["data"]

def get_channels(key):
    account=buffer_gql(key,ACCOUNT_QUERY).get("account")
    if not account:
        raise RuntimeError("Buffer account query returned no account.")
    result=[]
    for org in account.get("organizations",[]):
        data=buffer_gql(key,CHANNEL_QUERY,{"organizationId":org["id"]})
        for channel in data.get("channels",[]):
            if not channel.get("isDisconnected") and not channel.get("isLocked"):
                channel["organization_name"]=org.get("name","")
                result.append(channel)
    return result

def publish_video(key, channel, video_url, filename, title, description, tags):
    service=(channel.get("service") or "").lower()

    body_parts = []
    if description:
        body_parts.append(description)
    if tags:
        body_parts.append(tags)
    body_text = "\n\n".join(body_parts) if body_parts else ""

    common={"channelId":channel["id"],"text":body_text,"videoUrl":video_url}
    if "youtube" in service:
        yt_title = title if title else Path(filename).stem[:100]
        common.update({"title":yt_title,"categoryId":"22","privacy":"public"})
        return buffer_gql(key,YOUTUBE_MUTATION,common)
    if "instagram" in service:
        return buffer_gql(key,INSTAGRAM_MUTATION,common)
    return buffer_gql(key,GENERIC_MUTATION,common)

# ---------------- MAIN ----------------
def main():
    ensure_drive()
    os.makedirs(STATE_DIR,exist_ok=True)
    os.makedirs(LOG_DIR,exist_ok=True)

    rows=read_sheet()
    routes=build_routes(rows)

    # Discover virtual videos directly from metadata CSV files
    videos=discover_virtual_videos_from_csvs(routes)
    if MAX_VIDEOS > 0:
        videos=videos[:MAX_VIDEOS]

    state_file=os.path.join(STATE_DIR,"published_state.json")
    state=load_json(state_file,{})
    if not isinstance(state,dict): state={}

    log("="*68)
    log("BUFFER SHEET-ONLY VIDEO UPLOADER V2.8 (CSV DRIVEN)")
    log(f"Sheet routing folders: {', '.join(v['folder'] for v in routes.values())}")
    log(f"Virtual videos discovered: {len(videos)}")
    log(f"Publish immediately: {PUBLISH_NOW}")
    log("="*68)

    stats={ "videos":0, "published":0, "skipped":0, "failed":0 }
    log_file=os.path.join(LOG_DIR,"run_"+datetime.now().strftime("%Y%m%d_%H%M%S")+".json")
    events=[]

    for idx, video_info in enumerate(videos,1):
        stats["videos"]+=1
        filename = video_info["filename"]
        matched_category = video_info["folder_category"]
        video_url = video_info["video_url"]
        title = video_info["title"]
        description = video_info["description"]
        tags = video_info["tags"]

        # Create a stable identifier based on filename + URL for tracking publishing state
        unique_id = hashlib.sha256(f"{filename}:{video_url}".encode('utf-8')).hexdigest()[:16]

        log(f"[{idx}/{len(videos)}] {matched_category}/{filename}")
        log("-"*40)
        log("METADATA AND REMOTE LINK INFO:")
        log(f"  > Video Name : {filename}")
        log(f"  > Title      : '{title}'")
        log(f"  > Description: '{description}'")
        log(f"  > Tags       : '{tags}'")
        log(f"  > Video Link : '{video_url}'")
        log("-"*40)

        route=routes.get(matched_category.casefold())
        if not route:
            log("No matching sheet route; skipped.")
            stats["skipped"]+=1
            continue

        try:
            channel_jobs = {}
            for key_index, key in enumerate(route["keys"], 1):
                try:
                    channels = get_channels(key)
                    if not channels:
                        log(f"No connected Buffer channels for sheet route '{matched_category}' (key {key_index}).")
                        continue
                    for channel in channels:
                        channel_id = str(channel.get("id") or "").strip()
                        if not channel_id:
                            log(f"Channel without an ID on key {key_index}; skipped.")
                            stats["skipped"] += 1
                            continue
                        if channel_id not in channel_jobs:
                            channel_jobs[channel_id] = (key, channel)
                except Exception as e:
                    stats["failed"] += 1
                    events.append({"video":filename,"folder":matched_category,"api_key_index":key_index,"stage":"channel_discovery","ok":False,"error":str(e)})
                    log(f"Channel discovery error on sheet key {key_index}; continuing to other keys: {e}")

            if not channel_jobs:
                log(f"No usable connected channels found for '{matched_category}'.")
                continue

            for channel_id, (key, channel) in channel_jobs.items():
                service = (channel.get("service") or "unknown").lower()
                channel_name = channel.get("name") or channel.get("displayName") or channel_id
                state_key=f"{unique_id}:{channel_id}"

                if SKIP_ALREADY_PUBLISHED and state.get(state_key,{}).get("submitted"):
                    log(f"Already submitted to {channel_name}; skipped.")
                    stats["skipped"]+=1
                    continue

                try:
                    result = None
                    last_error = None
                    for attempt in range(MAX_CHANNEL_RETRIES + 1):
                        try:
                            result = publish_video(key, channel, video_url, filename, title, description, tags)
                            last_error = None
                            break
                        except Exception as e:
                            last_error = e
                            if attempt < MAX_CHANNEL_RETRIES:
                                log(f"Temporary error on {channel_name}; retrying once: {e}")
                                time.sleep(2)
                    if last_error is not None:
                        raise last_error

                    action = result.get("createPost",{}) if isinstance(result,dict) else {}
                    if action.get("message"):
                        raise RuntimeError(action["message"])
                    post=action.get("post") or {}
                    if not post.get("id"):
                        raise RuntimeError(f"Buffer returned no post ID: {str(action)[:300]}")

                    state[state_key]={"submitted":True,"time":now(),"video":filename,"channel":channel_name,"service":channel.get("service"),"post_id":post.get("id"),"status":post.get("status")}
                    save_json(state_file,state)
                    stats["published"]+=1
                    events.append({"video":filename,"folder":matched_category,"channel":channel_name,"service":channel.get("service"),"ok":True,"post_id":post.get("id"),"status":post.get("status")})
                    log(f"Submitted to Buffer: {channel_name} ({service})")
                except Exception as e:
                    stats["failed"]+=1
                    events.append({"video":filename,"folder":matched_category,"channel":channel_name,"service":channel.get("service"),"ok":False,"error":str(e)})
                    log(f"Publish error on {channel_name} ({service}); skipping: {e}")
        except Exception as e:
            stats["failed"]+=1
            events.append({"video":filename,"folder":matched_category,"ok":False,"error":str(e)})
            log(f"Video failed: {e}")
            traceback.print_exc()

    save_json(state_file,state)
    save_json(log_file,{"time":now(),"stats":stats,"events":events})
    log("="*68)
    log(f"DONE | videos={stats['videos']} submitted={stats['published']} skipped={stats['skipped']} failed={stats['failed']}")
    log(f"State: {state_file}")
    log(f"Run log: {log_file}")
    log("="*68)

if __name__ == "__main__":
    main()

[17:34:03] Scanning local metadata directory for CSVs: '/content/metadata_files'
[17:34:03]   Processing metadata file 'generated_metadata_Sad.csv' for Category 'Sad'
[17:34:03]     Found 1 virtual video link(s) inside 'generated_metadata_Sad.csv'
[17:34:03] ====================================================================
[17:34:03] BUFFER LOCAL-ONLY VIDEO UPLOADER V3.0
[17:34:03] Configured Buffer API Keys: 2
[17:34:03] Virtual videos discovered: 1
[17:34:03] Publish immediately: True
[17:34:03] ====================================================================
[17:34:03] [1/1] Sad/b0effae0326a0300941e6548284711a5_1793893555417.mp4
[17:34:03] ----------------------------------------
[17:34:03] METADATA AND REMOTE LINK INFO:
[17:34:03]   > Video Name : b0effae0326a0300941e6548284711a5_1793893555417.mp4
[17:34:03]   > Title      : 'Discover the Unknown'
[17:34:03]   > Description: 'Explore something new and intriguing in this video. What will you find?'
[17:34:03]   > Tags       :